# 06 - LoRA Task Vector Analysis

This notebook analyzes the task vectors of Logs and QA LoRA adapters.

The analysis includes:

- L2 norms of task vectors;
- cosine similarity between task directions;
- fraction of conflicting parameters;
- fraction of zero-valued parameters;
- tensor-level distributions of the above statistics.

Additionally, the notebook analyzes parameter sparsification introduced by:
- DARE;
- TIES.

In [ ]:
import exp_common as E
import pandas as pd, json, torch
import matplotlib.pyplot as plt
from pathlib import Path

RUN_SEEDS = [42, 123, 777]
DROP_RATES = [0.3, 0.5, 0.7]
TOP_KS = [0.1, 0.2, 0.5]

In [ ]:
all_rows = []

for seed in RUN_SEEDS:
    logs_ad = E.ADAPTERS_DIR / 'logs' / f'seed_{seed}' / 'adapter'
    qa_ad = E.ADAPTERS_DIR / 'qa' / f'seed_{seed}' / 'adapter'

    logs_file = logs_ad / 'adapter_model.safetensors'
    qa_file = qa_ad / 'adapter_model.safetensors'

    if not logs_file.exists():
        raise FileNotFoundError(
            f'Missing Logs adapter for seed {seed}: {logs_file}'
        )

    if not qa_file.exists():
        raise FileNotFoundError(
            f'Missing QA adapter for seed {seed}: {qa_file}'
        )

    prefix = E.ANALYSIS_DIR / f'task_vectors_seed_{seed}'

    df = E.adapter_vector_analysis(
        logs_ad,
        qa_ad,
        prefix,
    )

    df = df.copy()
    df['seed'] = seed
    all_rows.append(df)

    display(df.head())

vectors_df = pd.concat(
    all_rows,
    ignore_index=True,
)

vectors_df.head()

In [ ]:
summaries = []

for seed in RUN_SEEDS:
    summary_path = (
        E.ANALYSIS_DIR /
        f'task_vectors_seed_{seed}_summary.json'
    )

    if not summary_path.exists():
        raise FileNotFoundError(
            f'Missing task-vector summary for seed {seed}: {summary_path}'
        )

    obj = json.loads(
        summary_path.read_text(encoding='utf-8')
    )

    summaries.append({
        'seed': seed,
        'global_cosine': obj['global_cosine'],
        'global_conflict_frac': obj['global_conflict_frac'],
        'logs_l2': obj['logs']['l2'],
        'qa_l2': obj['qa']['l2'],
        'logs_zero_frac': obj['logs']['zero_frac'],
        'qa_zero_frac': obj['qa']['zero_frac'],
    })

summary_df = pd.DataFrame(summaries)

summary_df.to_csv(
    E.ANALYSIS_DIR / 'task_vectors_summary.csv',
    index=False,
)

summary_df

In [ ]:
# Tensor-level distribution visualizations
if not vectors_df.empty:
    for col in ['cosine', 'conflict_frac', 'logs_l2', 'qa_l2', 'logs_zero_frac', 'qa_zero_frac']:
        plt.figure(figsize=(7, 4))
        vectors_df[col].dropna().hist(bins=50)
        plt.title(col)
        plt.xlabel(col)
        plt.ylabel('count')
        out = E.ANALYSIS_DIR / f'{col}_hist.png'
        plt.tight_layout(); plt.savefig(out, dpi=160); plt.show()
        print('saved', out)

In [ ]:
# DARE and TIES zeroing analysis for a selected seed.
rows = []
for seed in RUN_SEEDS:
    logs_ad = E.ADAPTERS_DIR/'logs'/f'seed_{seed}'/'adapter'
    qa_ad = E.ADAPTERS_DIR/'qa'/f'seed_{seed}'/'adapter'
    if not (logs_ad/'adapter_model.safetensors').exists() or not (qa_ad/'adapter_model.safetensors').exists():
        continue
    states = [E.load_adapter_state(logs_ad), E.load_adapter_state(qa_ad)]
    for dr in DROP_RATES:
        for i, st in enumerate(states):
            d = E.dare_state(st, drop_rate=dr, seed=seed+i)
            vals = torch.cat([v.float().flatten() for v in d.values() if torch.is_floating_point(v)])
            rows.append({'seed': seed, 'method': 'DARE', 'adapter': ['logs','qa'][i], 'drop_rate': dr, 'top_k': None, 'zero_frac': float((vals==0).float().mean().item()), 'l2': float(torch.linalg.vector_norm(vals).item())})
    for tk in TOP_KS:
        m = E.ties_merge(states, top_k=tk, alpha=1.0)
        vals = torch.cat([v.float().flatten() for v in m.values() if torch.is_floating_point(v)])
        rows.append({'seed': seed, 'method': 'TIES', 'adapter': 'merged', 'drop_rate': None, 'top_k': tk, 'zero_frac': float((vals==0).float().mean().item()), 'l2': float(torch.linalg.vector_norm(vals).item())})
zero_df = pd.DataFrame(rows)
zero_df.to_csv(E.ANALYSIS_DIR/'zeroing_after_methods.csv', index=False)
zero_df